# Chapter 14 · When the pattern curves
**Deck 3 · Live demo.** A straight line can’t fit the sleep hump; one extra clue, sleep², bends it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "font.size": 12})

## 1 · Your prediction
Sleep 4, 6, 8, 10, 12 hours → 55, 70, 75, 70, 55 marks. **Marks for 7 hours of sleep?**

In [ ]:
my_prediction = 74

In [ ]:
sleep = np.array([4, 6, 8, 10, 12], dtype=float)
marks = np.array([55, 70, 75, 70, 55], dtype=float)

## 2 · The best straight line
Least squares (the exact answer gradient descent would reach).

In [ ]:
A_line = np.c_[sleep, np.ones_like(sleep)]
slope, base = np.linalg.lstsq(A_line, marks, rcond=None)[0]
line = slope * sleep + base
line_loss = np.mean((line - marks) ** 2)
print(f'best line: slope {slope:.3f}, base {base:.1f}, loss {line_loss:.1f}')

## 2b · How did we know? Subtract neighbours twice
Sleep goes up in equal steps of 2. Constant 1st differences mean a line; constant 2nd differences mean a square curve.

In [ ]:
marks_by_sleep = np.array([55, 70, 75, 70, 55])
d1 = np.diff(marks_by_sleep)
d2 = np.diff(d1)
print('1st differences:', d1, '→ not constant: not a line')
print('2nd differences:', d2, '→ constant: add sleep²; negative: bends down')
assert list(d1) == [15, 5, -5, -15] and list(d2) == [-10, -10, -10]
print('✓ the data asks for sleep² (and deck 1: hours 2, 4, 6 → 50, 60, 70 gives', np.diff([50, 60, 70]), ': a line)')

## 3 · Add sleep² as a clue
$$\hat{y} = w_1\,\text{sleep} + w_2\,\text{sleep}^2 + b$$ Still a weighted sum, just one more column.

In [ ]:
A_curve = np.c_[sleep, sleep ** 2, np.ones_like(sleep)]
w1, w2, b = np.linalg.lstsq(A_curve, marks, rcond=None)[0]
curve = lambda s: w1 * s + w2 * s ** 2 + b
curve_loss = np.mean((curve(sleep) - marks) ** 2)
print(f'curve: w1 = {w1:.2f}, w2 = {w2:.2f}, b = {b:.2f}, loss {curve_loss:.6f}')
print('7 hours →', round(curve(7), 2))

## 4 · PROOF

In [ ]:
assert abs(slope) < 1e-9 and abs(base - 65) < 1e-9 and round(line_loss, 6) == 70, 'flat line at 65, loss 70'
assert np.allclose([w1, w2, b], [20, -1.25, -5]), 'w1 = 20, w2 = −1.25, b = −5'
assert curve_loss < 1e-9 and round(curve(7), 2) == 73.75, 'loss 0 and 73.75 at 7 hours'
assert abs(my_prediction - curve(7)) <= 1.5, 'your guess was close'
print('✓ proof: line 65 / loss 70 · curve 73.75 / loss 0')

## 5 · Picture

In [ ]:
xs = np.linspace(3, 13, 100)
plt.scatter(sleep, marks, s=80, zorder=3, label='students')
plt.plot(xs, slope * xs + base, '--', label='best line (flat at 65)')
plt.plot(xs, curve(xs), label='with sleep² (curve)')
plt.scatter([7], [curve(7)], marker='*', s=150, zorder=4, label='Asha, 7 h → 73.75')
plt.xlabel('sleep per night (hours)'); plt.ylabel('marks'); plt.legend(); plt.show()

## 6 · Your turn
Add sleep³ as a fourth column. Does it help on these five students, or just add risk?

In [ ]:
# TODO

## Recap
A line can’t bend. Give the model a curved clue (sleep²) and the same method fits the hump. Neural networks will learn such clues by themselves.